In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
cd /content/gdrive/MyDrive/project

/content/gdrive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project


In [3]:
import os
import pandas as pd
import requests
from tqdm import tqdm

def download_datasets():
    # 创建tcdata文件夹（如果不存在）
    os.makedirs('tcdata', exist_ok=True)

    try:
        # 读取data_list.csv文件
        df = pd.read_csv('tcdata/mchar_data_list_0515.csv')
        print("CSV文件列名:", df.columns.tolist())
        print("发现的数据文件:", len(df))

        # 遍历每一行，下载数据集
        for index, row in tqdm(df.iterrows(), total=len(df), desc="下载进度"):
            try:
                # 使用正确的列名：link 和 file
                url = row['link']
                filename = row['file']

                filepath = os.path.join('tcdata', filename)

                # 如果文件已存在，跳过下载
                if os.path.exists(filepath):
                    print(f"文件已存在，跳过: {filename}")
                    continue

                print(f"正在下载: {filename}")

                # 下载文件
                response = requests.get(url, stream=True)
                response.raise_for_status()

                # 获取文件大小用于进度条
                total_size = int(response.headers.get('content-length', 0))

                # 下载并显示进度
                with open(filepath, 'wb') as f, tqdm(
                    desc=filename,
                    total=total_size,
                    unit='iB',
                    unit_scale=True,
                    unit_divisor=1024,
                    leave=False  # 不在完成后保留进度条
                ) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

                print(f"✓ 下载完成: {filename}")

            except KeyError as e:
                print(f"❌ CSV文件列名错误: {e}")
                break
            except Exception as e:
                print(f"❌ 下载失败 {filename}: {str(e)}")
                # 如果下载失败，删除可能不完整的文件
                if os.path.exists(filepath):
                    os.remove(filepath)

    except FileNotFoundError:
        print("❌ 找不到 tcdata/data_list.csv 文件")
    except Exception as e:
        print(f"❌ 读取CSV文件失败: {str(e)}")

# 执行下载
if __name__ == "__main__":
    download_datasets()

CSV文件列名: ['file', 'size', 'link']
发现的数据文件: 6


下载进度: 100%|██████████| 6/6 [00:00<00:00, 772.55it/s]

文件已存在，跳过: mchar_train.zip
文件已存在，跳过: mchar_train.json
文件已存在，跳过: mchar_val.zip
文件已存在，跳过: mchar_val.json
文件已存在，跳过: mchar_test_a.zip
文件已存在，跳过: mchar_sample_submit_A.csv
